**Nomes:** Patrícia da Silva Costa & Valdir de Souza Carvalho Neto

**Turma:** SI 241

In [ ]:
import time
import random

### Criação das funções auxiliares e das fases 1 e 2 do GRASP

In [ ]:
def solucao_vazia():
  return []

def obter_peso(item, itens):
  return itens[item]['peso']


def avaliar_valor(solucao, itens):
  return sum(itens[item]['valor'] for item in solucao)


def avaliar_peso_total(solucao, itens):
  return sum(itens[item]['peso'] for item in solucao)


def calcular_beneficio_guloso(e, itens):
  return itens[e]['valor'] / itens[e]['peso']


def adicionar_item_na_solucao(solucao, item_escolhido):
  solucao.append(item_escolhido)

def obter_candidatos_viaveis(itens, solucao, capacidade_restante):
  return [
      item
      for item, dados in itens.items()
      if item not in solucao and dados['peso'] <= capacidade_restante
  ]

def gerar_vizinhanca_viavel(solucao_atual, itens, capacidade_maxima):
  """
  1) Trocar 1 item de dentro por 1 item de fora e se sobrar espaço
     insere mais 1 item de fora para aproveitar a capacidade liberada.
  """
  vizinhos = []
  itens_fora = [i for i in itens.keys() if i not in solucao_atual]

  # Remover 1 item da mochila
  for item_saindo in solucao_atual:
    solucao_sem_item = [i for i in solucao_atual if i != item_saindo]
    peso_base = avaliar_peso_total(solucao_sem_item, itens)

    # Tenta colocar 1 item que estava fora
    for idx, item_entrando_1 in enumerate(itens_fora):
      novo_peso_1 = peso_base + itens[item_entrando_1]['peso']
      if novo_peso_1 <= capacidade_maxima:
        vizinho_1 = solucao_sem_item + [item_entrando_1]
        vizinhos.append(vizinho_1)

        # Se sobrou capacidade após a troca encaixar um 2° item de fora
        for item_entrando_2 in itens_fora[idx + 1 :]:
          if novo_peso_1 + itens[item_entrando_2]['peso'] <= capacidade_maxima:
            vizinho_2 = vizinho_1 + [item_entrando_2]
            vizinhos.append(vizinho_2)

  return vizinhos

# Fase 1
def construcao_gulosa_aleatoria_adaptativa(itens, capacidade_maxima, alfa):
  solucao = solucao_vazia()
  peso_atual = 0

  candidatos = obter_candidatos_viaveis(
      itens, solucao, capacidade_maxima - peso_atual
  )

  while len(candidatos) > 0:
    # 1. Avalia o benefício incremental g(e) para todo item 'e' em C
    beneficios = {e: calcular_beneficio_guloso(e, itens) for e in candidatos}

    g_min = min(beneficios.values())
    g_max = max(beneficios.values())

    # 2. Cria a Lista Restrita de Candidatos (RCL) para MAXIMIZAÇÃO
    limiar = g_max - alfa * (g_max - g_min)
    # (Usamos - 1e-9 para evitar erros de arredondamento de ponto flutuante)
    RCL = [e for e in candidatos if beneficios[e] >= limiar - 1e-9]

    # 3. Seleciona um elemento aleatoriamente da RCL
    item_escolhido = random.choice(RCL)

    # 4. Adiciona o item à mochila e soma seu peso
    adicionar_item_na_solucao(solucao, item_escolhido)
    peso_atual += obter_peso(item_escolhido, itens)

    # 5. Fase Adaptativa: Atualiza a Lista de Candidatos C
    capacidade_restante = capacidade_maxima - peso_atual
    candidatos = obter_candidatos_viaveis(itens, solucao, capacidade_restante)

  return solucao

# Fase 2: Busca Local
def busca_local(solucao_inicial, itens, capacidade_maxima):
  solucao_atual = solucao_inicial.copy()
  melhorou = True

  while melhorou:
    melhorou = False
    valor_atual = avaliar_valor(solucao_atual, itens)

    vizinhos_viaveis = gerar_vizinhanca_viavel(
        solucao_atual, itens, capacidade_maxima
    )

    for vizinho in vizinhos_viaveis:
      valor_vizinho = avaliar_valor(vizinho, itens)
      if valor_vizinho > valor_atual:
        solucao_atual = vizinho
        melhorou = True
        break

  return solucao_atual

# principal
def grasp_mochila(
    itens, capacidade_maxima, max_iteracoes, semente_aleatoria, alfa
):
  random.seed(semente_aleatoria)
  melhor_solucao = None
  melhor_valor = float('-inf')

  for k in range(1, max_iteracoes + 1):
    solucao = construcao_gulosa_aleatoria_adaptativa(
        itens, capacidade_maxima, alfa
    )
    solucao = busca_local(solucao, itens, capacidade_maxima)
    valor_atual = avaliar_valor(solucao, itens)

    if valor_atual > melhor_valor:
      melhor_solucao = solucao.copy()
      melhor_valor = valor_atual

  return melhor_solucao, melhor_valor

### Atividade - Instância 1: mochila com 15 itens

In [ ]:
itens1 = {
  'A': {'peso': 2, 'valor': 12},
  'B': {'peso': 3, 'valor': 14},
  'C': {'peso': 5, 'valor': 25},
  'D': {'peso': 7, 'valor': 30},
  'E': {'peso': 8, 'valor': 32},
  'F': {'peso': 4, 'valor': 20},
  'G': {'peso': 1, 'valor': 4},
  'H': {'peso': 2, 'valor': 9},
  'I': {'peso': 3, 'valor': 13},
  'J': {'peso': 4, 'valor': 18},
  'K': {'peso': 5, 'valor': 22},
  'L': {'peso': 6, 'valor': 25},
  'M': {'peso': 7, 'valor': 29},
  'N': {'peso': 8, 'valor': 31},
  'O': {'peso': 9, 'valor': 36}
}

capacidade_maxima = 25

In [ ]:
num_execucoes = 20
max_iteracoes_grasp = 20

valores_solucoes1 = []
tempos_execucao1 = []

melhor_sol_global1 = None
melhor_val_global1 = float('-inf')
tempo_melhor_sol1 = 0

for i in range(num_execucoes):
    inicio = time.perf_counter()

    sol_atual, val_atual = grasp_mochila(
        itens=itens1,
        capacidade_maxima=capacidade_maxima,
        max_iteracoes=max_iteracoes_grasp,
        semente_aleatoria=42 + i,
        alfa=0.3,
    )

    fim = time.perf_counter()
    tempo_rodada = fim - inicio

    valores_solucoes1.append(val_atual)
    tempos_execucao1.append(tempo_rodada)

    if val_atual > melhor_val_global1:
        melhor_val_global1 = val_atual
        melhor_sol_global1 = sol_atual.copy()
        tempo_melhor_sol1 = tempo_rodada

media_valores_1 = sum(valores_solucoes1) / num_execucoes
media_tempos_1 = sum(tempos_execucao1) / num_execucoes
peso_final_1 = avaliar_peso_total(melhor_sol_global1, itens1)

print(f"Melhor solução encontrada: {sorted(melhor_sol_global1)}")
print(f"Peso total: {peso_final_1}/{capacidade_maxima}")
print(f"Valor solução: {melhor_val_global1} (Referência: 120)")
print(f"Tempo de execução da melhor solução: {tempo_melhor_sol1:.6f} segundos\n")
print("-"*10 + " Médias " + "-"*10)
print(f"Média dos valores das soluções: {media_valores_1:.2f}")
print(f"Média dos tempos de execução: {media_tempos_1:.6f} segundos")
print(f"Tempo total das 20 execuções: {sum(tempos_execucao1):.6f} segundos")

Melhor solução encontrada: ['A', 'B', 'C', 'F', 'H', 'J', 'K']
Peso total: 25/25
Valor solução: 120 (Referência: 120)
Tempo de execução da melhor solução: 0.004851 segundos

---------- Médias ----------
Média dos valores das soluções: 120.00
Média dos tempos de execução: 0.004099 segundos
Tempo total das 20 execuções: 0.081977 segundos


### Atividade - Instância 2: mochila com 20 itens

In [ ]:
itens2 = {
    'A': {'peso': 3, 'valor': 7},
    'B': {'peso': 5, 'valor': 12},
    'C': {'peso': 6, 'valor': 14},
    'D': {'peso': 7, 'valor': 18},
    'E': {'peso': 9, 'valor': 22},
    'F': {'peso': 11, 'valor': 27},
    'G': {'peso': 12, 'valor': 29},
    'H': {'peso': 14, 'valor': 34},
    'I': {'peso': 15, 'valor': 36},
    'J': {'peso': 17, 'valor': 40},
    'K': {'peso': 2, 'valor': 5},
    'L': {'peso': 4, 'valor': 10},
    'M': {'peso': 8, 'valor': 20},
    'N': {'peso': 10, 'valor': 24},
    'O': {'peso': 13, 'valor': 31},
    'P': {'peso': 16, 'valor': 38},
    'Q': {'peso': 18, 'valor': 42},
    'R': {'peso': 20, 'valor': 47},
    'S': {'peso': 22, 'valor': 52},
    'T': {'peso': 24, 'valor': 57}
}

capacidade_maxima2 = 55

In [ ]:
num_execucoes = 20
max_iteracoes_grasp = 20

valores_solucoes2 = []
tempos_execucao2 = []

melhor_sol_global2 = None
melhor_val_global2 = float('-inf')
tempo_melhor_sol2 = 0

for i in range(num_execucoes):
    inicio2 = time.perf_counter()

    sol_atual2, val_atual2 = grasp_mochila(
        itens=itens2,
        capacidade_maxima=capacidade_maxima2,
        max_iteracoes=max_iteracoes_grasp,
        semente_aleatoria=10 + i,
        alfa=0.3,
    )

    fim2 = time.perf_counter()
    tempo_rodada2 = fim2 - inicio2

    valores_solucoes2.append(val_atual2)
    tempos_execucao2.append(tempo_rodada2)

    if val_atual2 > melhor_val_global2:
        melhor_val_global2 = val_atual2
        melhor_sol_global2 = sol_atual2.copy()
        tempo_melhor_sol2 = tempo_rodada2

media_valores_2 = sum(valores_solucoes2) / num_execucoes
media_tempos_2 = sum(tempos_execucao2) / num_execucoes
peso_final_2 = avaliar_peso_total(melhor_sol_global2, itens2)

print(f"Melhor solução encontrada: {sorted(melhor_sol_global2)}")
print(f"Peso total: {peso_final_2} / {capacidade_maxima2}")
print(f"Valor da solução: {melhor_val_global2} (Referência: 136)")
print(f"Tempo de execução da melhor solução: {tempo_melhor_sol2:.6f} segundos\n")
print("-"*10 + " Médias " + "-"*10)
print(f"Média dos valores das soluções: {media_valores_2:.2f}")
print(f"Média dos tempos de execução: {media_tempos_2:.6f} segundos")
print(f"Tempo total das 20 execuções: {sum(tempos_execucao2):.6f} segundos")

Melhor solução encontrada: ['D', 'E', 'F', 'H', 'K', 'L', 'M']
Peso total: 55 / 55
Valor da solução: 136 (Referência: 136)
Tempo de execução da melhor solução: 0.005532 segundos

---------- Médias ----------
Média dos valores das soluções: 136.00
Média dos tempos de execução: 0.006634 segundos
Tempo total das 20 execuções: 0.132680 segundos
